# 01_09_remove_wcs_in_HDUL_ys_all

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [2]:
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

This notebook was generated at 2023-02-01 18:57:31 (KST = GMT+0900) 
0 Python     3.8.16 64bit [GCC 11.2.0]
1 IPython    8.8.0
2 OS         Linux 5.15.0 58 generic x86_64 with glibc2.17
3 numpy      1.22.3
4 pandas     1.5.2
5 matplotlib 3.6.3
6 scipy      1.8.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 version_information 1.0.4


### import modules

In [11]:
import os, shutil
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.stats import sigma_clip
from astropy.io import fits
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

from snuo1mpy import Preprocessor

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [12]:
#%%
BASEDIR = Path(astro_utilities.CCD_obs_raw_dir)

fpaths = sorted(list(BASEDIR.glob("*.csv")))
print(fpaths)

for summary_fpath in fpaths :
    summary_all = pd.read_csv(str(summary_fpath))
    print(summary_all)


[PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_2bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_3bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/ST-8300M_1bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/ST-8300M_2bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_2bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STL-11000M_1bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STL-11000M_2bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_1bin.csv'), PosixPath('/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin.csv')]


/tmp/ipykernel_117786/3480723369.py:8: DtypeWarning: Columns (22,23,28,31,38,39,54,61,62,64,66,67,68,72,78,79,80,81,87,88,90,92,97,98,99,101,102,103,110,111,112,147,152,153,154,155,157,158,163,164,169,170,171,172,173,175,179,180,181,182,185,186,192) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_all = pd.read_csv(str(summary_fpath))
/tmp/ipykernel_117786/3480723369.py:8: DtypeWarning: Columns (41,59,61,62,63,82,84,92,127,130,131,133,134,139,140,149) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_all = pd.read_csv(str(summary_fpath))


       Unnamed: 0  index                                               file  \
0               0      0  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/C...   
1               1      1  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/C...   
2               2      2  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/C...   
3               3      3  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/C...   
4               4      4  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/C...   
...           ...    ...                                                ...   
24831       24831     79  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/L...   
24832       24832     80  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/L...   
24833       24833     81  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/L...   
24834       24834     82  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/L...   
24835       24835     83  /mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/L...   

       filesize  SIMPLE  BITPIX  NAXIS  NAXIS1  NAX

/tmp/ipykernel_117786/3480723369.py:8: DtypeWarning: Columns (38,54,58,63,66,70,73,74,75,76,85,86,87,88,94,97,99,108,109,111,112,113,114,115,116,117,118,120,121,122,123,130,131,145,148,149,183,184,185,186,187,188,191,192,193,194,195,196,197,198) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_all = pd.read_csv(str(summary_fpath))
/tmp/ipykernel_117786/3480723369.py:8: DtypeWarning: Columns (73,74,75,93,114,115,120) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_all = pd.read_csv(str(summary_fpath))


       Unnamed: 0  index                                               file  \
0               0      0  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
1               1      1  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
2               2      2  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
3               3      3  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
4               4      4  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
...           ...    ...                                                ...   
30173       30173     75  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
30174       30174     76  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
30175       30175     77  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
30176       30176     78  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   
30177       30177     79  /mnt/Rdata/CCD_obs/CCD_obs_raw/STF-8300M_1bin/...   

       filesize  SIMPLE  BITPIX  NAXIS  NAXIS1  NAX

/tmp/ipykernel_117786/3480723369.py:8: DtypeWarning: Columns (20,26,28,33,36,38,49,50,51,53,54,57,69,78,86,105,106,112,143,146,147,149,150,154,155,159) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_all = pd.read_csv(str(summary_fpath))
/tmp/ipykernel_117786/3480723369.py:9: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  print(summary_all)
/tmp/ipykernel_117786/3480723369.py:9: FutureWarning: In a future version, object-dtype columns with all-bool values will not be included in reductions with bool_only=True. Explicitly cast to bool dtype instead.
  print(summary_all)


In [13]:
for _, row in summary_all.iterrows():
    # 파일명 출력
    print (row["file"])
    fpath = Path(row["file"])
    new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_clean.fit")
    # fits hedaer 에 있는 wcs 정보를 지운다
    yfu.wcsremove(fpath, 
                    additional_keys=["COMMENT"],
                    verbose=True,
                    output=new_fpath,
                    ccddata=True,
                    overwrite=True)
    if new_fpath.exists() \
            and fpath.exists():
            print("rename", f"{str(new_fpath)}", f"{str(fpath)}")
            #os.rename(f"{str(new_fpath)}", f"{str(fpath)}")
            shutil.move(f"{str(new_fpath)}", f"{str(fpath)}")
            
    # except Exception as err :
    #     print("X"*60)
    #     print('{0}'.format(err))

/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-32_000sec_RiLA600_STX-16803_-10C_2bin.fit
Removed keywords: 
COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT 

DELMAG   =   0.0820            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]
FWHMHS   =   0.0000            /Horizontal FWHM std dev, pixels                  [astropy.io.fits.card]
FWHMVS   =   0.0000            /Vertical FWHM std dev, pixels                    [astropy.io.fits.card]


rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-32_000sec_RiLA600_STX-16803_-10C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-32_000sec_RiLA600_STX-16803_-10C_2bin.fit
/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-39_000sec_RiLA600_STX-16803_-9C_2bin.fit
Removed keywords: 
COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-39_000sec_RiLA600_STX-16803_-9C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2016-03-30_-_-_STX-16803_-_2bin/-_Bias_-_2016-03-30-10-15-39_000sec_RiLA600_STX-16803_-9C_2bin.fit
/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_

DELMAG   =   0.0000            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2017-02-19_-_-_STX-16803_-_2bin/-_Bias_-_2017-02-19-17-25-07_000sec_RiLA600_STX-16803_-19C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2017-02-19_-_-_STX-16803_-_2bin/-_Bias_-_2017-02-19-17-25-07_000sec_RiLA600_STX-16803_-19C_2bin.fit
/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2017-02-19_-_-_STX-16803_-_2bin/-_Bias_-_2017-02-19-17-25-14_000sec_RiLA600_STX-16803_-19C_2bin.fit
Removed keywords: 
COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2017-02-19_-_-_STX-16803_-_2bin/-_Bias_-_2017-02-19-17-25-14_000sec_RiLA600_STX-16803_-19C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias_-_2017-02-19_-_-_STX-16803_-_2bin/-_Bias_-_2017-02-19-17-25-14_000sec_RiLA600_STX-16803_-19C_2bin.fit
/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Bias

DELMAG   =  -0.1111            /Veriable - Comparison,   Delmag                  [astropy.io.fits.card]


WCSDIM LTM1_1 LTM2_2 WAT0_001 WAT1_001 WAT2_001 COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Dark_-_2016-03-30_100sec_-_STX-16803_-_2bin/-_Dark_-_2016-03-30-22-11-23_100sec_RiLA600_STX-16803_-10C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Dark_-_2016-03-30_100sec_-_STX-16803_-_2bin/-_Dark_-_2016-03-30-22-11-23_100sec_RiLA600_STX-16803_-10C_2bin.fit
/mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Dark_-_2016-03-30_100sec_-_STX-16803_-_2bin/-_Dark_-_2016-03-30-22-13-10_100sec_RiLA600_STX-16803_-10C_2bin.fit
Removed keywords: 
COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT COMMENT rename /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2bin/Cal/-_Dark_-_2016-03-30_100sec_-_STX-16803_-_2bin/-_Dark_-_2016-03-30-22-13-10_100sec_RiLA600_STX-16803_-10C_2bin_clean.fit /mnt/Rdata/CCD_obs/CCD_obs_raw/STX-16803_2b